In [1]:
# Bibliotheken importieren
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Datensatz laden
df = pd.read_csv('../data/adult.csv')

# Ersten Überblick verschaffen
print(df.shape)
print(df.head())


(32561, 15)
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356              18  U

In [2]:
# Bibliotheken importieren
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Datensatz laden
df = pd.read_csv('../data/adult.csv')

# Ersten Überblick verschaffen
print(df.shape)
print(df.head())


(32561, 15)
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income  
0          4356              40  United-States  <=50K  
1          4356              18  U

In [3]:
# Grundlegende Infos zum Datensatz
print(df.dtypes)
print("---")
print(df['income'].value_counts())
print("---")
print(df['sex'].value_counts())


age                int64
workclass         object
fnlwgt             int64
education         object
education.num      int64
marital.status    object
occupation        object
relationship      object
race              object
sex               object
capital.gain       int64
capital.loss       int64
hours.per.week     int64
native.country    object
income            object
dtype: object
---
income
<=50K    24720
>50K      7841
Name: count, dtype: int64
---
sex
Male      21790
Female    10771
Name: count, dtype: int64


In [4]:
# Fehlende Werte prüfen
print(df.isnull().sum())
print("---")
# Fragezeichen als fehlende Werte zählen
for col in df.select_dtypes(include='object').columns:
    n = (df[col] == '?').sum()
    if n > 0:
        print(f"{col}: {n} Fragezeichen")

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64
---
workclass: 1836 Fragezeichen
occupation: 1843 Fragezeichen
native.country: 583 Fragezeichen


In [5]:
# Fragezeichen durch häufigsten Wert ersetzen
df_clean = df.copy()

for col in ['workclass', 'occupation', 'native.country']:
    most_frequent = df_clean[col][df_clean[col] != '?'].mode()[0]
    df_clean[col] = df_clean[col].replace('?', most_frequent)
    print(f"{col}: Fragezeichen ersetzt durch '{most_frequent}'")

# Kontrolle
print("---")
print("Verbleibende Fragezeichen:", (df_clean == '?').sum().sum())

workclass: Fragezeichen ersetzt durch 'Private'
occupation: Fragezeichen ersetzt durch 'Prof-specialty'
native.country: Fragezeichen ersetzt durch 'United-States'
---
Verbleibende Fragezeichen: 0


In [6]:
# Geschlechtsmerkmal als separate Variable speichern
sex = df_clean['sex'].map({'Male': 1, 'Female': 0})

# Zielmerkmal kodieren
y = df_clean['income'].map({'<=50K': 0, '>50K': 1})

# Geschlecht, Zielmerkmal und fnlwgt aus den Features entfernen
X = df_clean.drop(columns=['sex', 'income', 'fnlwgt'])

print("Features:", X.columns.tolist())
print("Shape X:", X.shape)
print("Shape y:", y.shape)


Features: ['age', 'workclass', 'education', 'education.num', 'marital.status', 'occupation', 'relationship', 'race', 'capital.gain', 'capital.loss', 'hours.per.week', 'native.country']
Shape X: (32561, 12)
Shape y: (32561,)


In [7]:
# Kategoriale Merkmale kodieren
X_encoded = pd.get_dummies(X)

print("Shape nach Kodierung:", X_encoded.shape)

# Train-Test-Split 80/20
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, sex_train, sex_test = train_test_split(
    X_encoded, y, sex, test_size=0.2, random_state=42
)

print("Trainingsdaten:", X_train.shape)
print("Testdaten:", X_test.shape)


Shape nach Kodierung: (32561, 102)
Trainingsdaten: (26048, 102)
Testdaten: (6513, 102)


In [8]:
# Train-Test-Split 80/20
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, sex_train, sex_test = train_test_split(
    X_encoded, y, sex, test_size=0.2, random_state=42
)

print("Trainingsdaten:", X_train.shape)
print("Testdaten:", X_test.shape)

Trainingsdaten: (26048, 102)
Testdaten: (6513, 102)


In [9]:
# Vorverarbeiteten Datensatz speichern
X_train.to_csv('../data/X_train.csv', index=False)
X_test.to_csv('../data/X_test.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)
sex_train.to_csv('../data/sex_train.csv', index=False)
sex_test.to_csv('../data/sex_test.csv', index=False)

print("Alle Dateien gespeichert.")

Alle Dateien gespeichert.


In [10]:
# race separat speichern (analog zu sex)
race = df_clean['race']

_, _, _, _, race_train, race_test = train_test_split(
    X_encoded, y, race, test_size=0.2, random_state=42
)

race_train.to_csv('../data/race_train.csv', index=False)
race_test.to_csv('../data/race_test.csv', index=False)
print("race_train und race_test gespeichert.")
print(race_test.value_counts())

race_train und race_test gespeichert.
race
White                 5568
Black                  625
Asian-Pac-Islander     201
Amer-Indian-Eskimo      65
Other                   54
Name: count, dtype: int64
